# Bug Triage Agent

A multi-step bug triage agent that investigates bug reports using tools and classifies them.  
Instrumented with Arize AX for observability and evaluation.

**Fictional product:** TaskFlow AI — a B2B project management SaaS  
**Agent pattern:** Raw agentic loop (no framework) with Anthropic tool use  
**Tracing:** `arize-otel` + OpenTelemetry manual spans

## 1. Install Dependencies

In [ ]:
%pip install anthropic arize-otel openinference-instrumentation-anthropic openinference-semantic-conventions python-dotenv --quiet

## 2. Imports & Configuration

In [ ]:
import os
import json
import anthropic
from dotenv import load_dotenv
from arize.otel import register
from opentelemetry import trace
from opentelemetry.trace import Status, StatusCode

load_dotenv()

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
ARIZE_SPACE_ID    = os.environ["ARIZE_SPACE_ID"]
ARIZE_API_KEY     = os.environ["ARIZE_API_KEY"]

MODEL = "claude-opus-4-6"

print("Keys loaded successfully")

## 3. Arize Tracing Setup

In [ ]:
tracer_provider = register(
    space_id=ARIZE_SPACE_ID,
    api_key=ARIZE_API_KEY,
    project_name="bug-triage-agent",
)

# No AnthropicInstrumentor — we manually create LLM spans for full control over nesting
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Get tracer from the Arize-registered provider directly
tracer = tracer_provider.get_tracer(__name__)

print("Arize tracing initialized. Project: bug-triage-agent")

## 4. Simulated Data

All data is simulated — no live database or API calls required to run this notebook.

In [ ]:
# --- Product Documentation ---
PRODUCT_DOCS = {
    "export_api": {
        "title": "Data Export API",
        "content": (
            "The /api/v2/export endpoint supports CSV and JSON formats. "
            "Include the format parameter (?format=csv or ?format=json). "
            "Maximum 10,000 rows per request. Requests exceeding this limit "
            "return HTTP 413 with error code EXPORT_LIMIT_EXCEEDED. "
            "For larger exports, use the /api/v2/export/bulk endpoint which "
            "generates a downloadable file and sends a notification when ready. "
            "Exports require Member role or above. Guest users receive HTTP 403."
        ),
        "last_updated": "2026-03-15",
        "tags": ["api", "export", "csv", "json", "data"]
    },
    "permissions_roles": {
        "title": "Roles and Permissions",
        "content": (
            "TaskFlow AI uses role-based access control with four roles: "
            "Owner (full access, billing, can delete workspace), "
            "Admin (manage members, configure integrations, manage all projects), "
            "Member (create/edit tasks in assigned projects, export data, use AI features), "
            "Guest (view-only access to specific projects they are invited to, cannot export, "
            "cannot use AI features). "
            "Guests cannot access the API. Guest users attempting API calls receive HTTP 403. "
            "Role changes take effect immediately but active sessions may need to re-authenticate."
        ),
        "last_updated": "2026-02-01",
        "tags": ["permissions", "roles", "access", "guest", "admin"]
    },
    "ai_task_summarizer": {
        "title": "AI Task Summarizer",
        "content": (
            "The AI Task Summarizer generates concise summaries of task activity. "
            "Triggered manually via the 'Summarize' button or automatically for tasks "
            "with 5 or more comments. Summaries include key decisions, action items, "
            "and current status. The summarizer processes the most recent 50 comments "
            "and attached text files (PDFs and images are not processed). "
            "Summaries are regenerated each time — they are not cached. "
            "The feature is available to Member and Admin roles only. "
            "Known limitation: the summarizer may not distinguish between resolved "
            "and unresolved discussion threads, which can lead to outdated action items "
            "appearing in summaries."
        ),
        "last_updated": "2026-04-01",
        "tags": ["ai", "summarizer", "summary", "tasks", "comments"]
    },
    "smart_auto_assign": {
        "title": "Smart Auto-Assign",
        "content": (
            "Smart Auto-Assign recommends or automatically assigns team members to new tasks. "
            "Two modes: 'suggest' (shows a recommendation the creator can accept/reject) and "
            "'auto' (assigns immediately on task creation). "
            "Assignment logic considers: current workload (open task count), skill tags on "
            "user profiles, past assignment patterns in the same project, and sprint capacity. "
            "If no suitable assignee is found, the task is left unassigned with a note. "
            "Auto-assign runs only on task creation — it does not reassign existing tasks. "
            "The algorithm refreshes workload data every 15 minutes, so recently completed "
            "tasks may not be reflected immediately. "
            "Requires Admin to enable per-project in Project Settings > Automation."
        ),
        "last_updated": "2026-03-20",
        "tags": ["ai", "auto-assign", "assignment", "automation", "workload"]
    },
    "ai_weekly_digest": {
        "title": "AI Weekly Digest",
        "content": (
            "The AI Weekly Digest sends an automated status report every Monday at 9:00 AM "
            "in the workspace's configured timezone. Each project with activity in the past "
            "7 days gets a section covering: completed tasks, new blockers, upcoming deadlines "
            "(next 7 days), and a velocity comparison to the previous week. "
            "The digest is sent to all Members and Admins via email and posted to the "
            "configured Slack channel if the Slack integration is active. "
            "Digest generation typically takes 2-5 minutes. If a project has more than "
            "200 tasks modified in the past week, the digest summarizes by category "
            "rather than listing individual tasks. "
            "Digests can be disabled per-project in Project Settings > Notifications."
        ),
        "last_updated": "2026-03-25",
        "tags": ["ai", "digest", "weekly", "report", "status", "notifications"]
    },
    "file_attachments": {
        "title": "File Attachments",
        "content": (
            "Files can be attached to tasks and comments. Maximum file size: 50MB. "
            "Maximum storage per workspace: 500MB (Starter), 5GB (Pro), Unlimited (Enterprise). "
            "Supported preview formats: PNG, JPG, GIF, PDF, MP4 (under 100MB). "
            "Other file types are downloadable but not previewable. "
            "Files are stored on AWS S3 with server-side encryption. "
            "Deleting a task moves attachments to a 30-day trash. "
            "Attachment URLs are signed and expire after 1 hour. "
            "If a file download link returns 403, the URL has likely expired — "
            "reload the task to generate a fresh link."
        ),
        "last_updated": "2026-01-15",
        "tags": ["files", "attachments", "upload", "storage", "s3"]
    },
    "sso_authentication": {
        "title": "SSO / SAML Authentication",
        "content": (
            "TaskFlow AI supports SAML 2.0 SSO with Okta and Azure AD. "
            "SSO is available on Pro and Enterprise plans. "
            "When SSO is enabled, users are redirected to the identity provider "
            "on login. After authentication, they are redirected to /auth/sso/callback "
            "and then to their default project dashboard. "
            "Session duration is 24 hours by default, configurable by Admin up to 72 hours. "
            "If a user reports a redirect loop, check that the ACS URL in the identity "
            "provider matches https://app.taskflow.ai/auth/sso/callback exactly. "
            "Trailing slashes or http:// (instead of https://) will cause loops."
        ),
        "last_updated": "2026-02-10",
        "tags": ["sso", "saml", "authentication", "login", "okta", "azure"]
    },
    "webhooks_integrations": {
        "title": "Webhooks and Integrations",
        "content": (
            "TaskFlow AI supports outgoing webhooks to Slack, GitHub, and Jira. "
            "Webhooks fire on: task created, task status changed, comment added, "
            "sprint started/completed. "
            "Webhook payloads are JSON with a 5-second timeout. Failed deliveries "
            "are retried 3 times with exponential backoff (10s, 30s, 90s). "
            "After 3 failures, the webhook is marked 'degraded' and an Admin notification "
            "is sent. Webhooks can be re-enabled manually in Settings > Integrations. "
            "The Slack integration uses OAuth and requires re-authorization if the "
            "Slack workspace admin revokes the app."
        ),
        "last_updated": "2026-03-01",
        "tags": ["webhooks", "slack", "github", "jira", "integrations"]
    },
    "dashboards_charts": {
        "title": "Team Dashboards",
        "content": (
            "Dashboards display burndown charts, velocity graphs, and task distribution "
            "by status, assignee, or label. Data refreshes every 10 minutes. "
            "Custom date ranges are supported (up to 90 days). "
            "Dashboard widgets can be rearranged via drag-and-drop. "
            "Export dashboard as PNG or PDF via the share menu. "
            "Note: dashboard data reflects task status at the time of the last refresh — "
            "very recent changes (within the past 10 minutes) may not appear. "
            "If a chart shows 'No data available,' ensure the project has tasks with "
            "the required fields populated (e.g., story points for velocity charts)."
        ),
        "last_updated": "2026-02-20",
        "tags": ["dashboard", "charts", "burndown", "velocity", "reporting"]
    },
    "sprint_planning": {
        "title": "Sprint Planning",
        "content": (
            "Sprints are created in the Planning view with a start date, end date, "
            "and optional sprint goal. Default sprint duration is 2 weeks. "
            "Tasks are added to sprints by dragging from the backlog or using the "
            "'Add to Sprint' action. Sprint capacity is calculated from team member "
            "availability (set in Team Settings). "
            "Active sprint cannot be deleted — it must be completed or cancelled. "
            "Completing a sprint moves unfinished tasks to the backlog unless "
            "'auto-move to next sprint' is enabled in Project Settings. "
            "Sprint velocity is calculated from story points of completed tasks only."
        ),
        "last_updated": "2026-01-30",
        "tags": ["sprint", "planning", "backlog", "velocity", "capacity"]
    }
}

print(f"Loaded {len(PRODUCT_DOCS)} product documentation entries")

In [ ]:
# --- Error Logs ---
ERROR_LOGS = [
    {
        "timestamp": "2026-05-06T14:23:01Z",
        "level": "ERROR",
        "service": "export-service",
        "message": "Export request failed: EXPORT_LIMIT_EXCEEDED. Requested 24,891 rows, limit is 10,000. user_id=8842, workspace_id=ws_301, endpoint=/api/v2/export?format=csv",
        "trace_id": "exp-001",
        "request_id": "req-44a1"
    },
    {
        "timestamp": "2026-05-06T14:23:02Z",
        "level": "WARN",
        "service": "export-service",
        "message": "Bulk export fallback not triggered — user did not use /api/v2/export/bulk endpoint. user_id=8842",
        "trace_id": "exp-001",
        "request_id": "req-44a2"
    },
    {
        "timestamp": "2026-05-06T09:15:33Z",
        "level": "ERROR",
        "service": "ai-summarizer",
        "message": "Summarizer output flagged by post-processing check: summary references task TF-892 which was deleted 3 days ago. project_id=proj_12, task_id=TF-1204",
        "trace_id": "sum-002",
        "request_id": "req-55b1"
    },
    {
        "timestamp": "2026-05-06T09:15:34Z",
        "level": "INFO",
        "service": "ai-summarizer",
        "message": "Summary generated in 4.2s. Input: 38 comments, 2 text attachments. Output: 156 words. task_id=TF-1204",
        "trace_id": "sum-002",
        "request_id": "req-55b2"
    },
    {
        "timestamp": "2026-05-06T11:02:47Z",
        "level": "ERROR",
        "service": "auth-service",
        "message": "SSO redirect loop detected: 3 redirects in 10 seconds. user_id=2201, idp=okta, acs_url=http://app.taskflow.ai/auth/sso/callback",
        "trace_id": "auth-003",
        "request_id": "req-66c1"
    },
    {
        "timestamp": "2026-05-06T11:02:48Z",
        "level": "WARN",
        "service": "auth-service",
        "message": "ACS URL mismatch: configured='http://app.taskflow.ai/auth/sso/callback', expected='https://app.taskflow.ai/auth/sso/callback'. Possible http/https mismatch.",
        "trace_id": "auth-003",
        "request_id": "req-66c2"
    },
    {
        "timestamp": "2026-05-05T16:45:12Z",
        "level": "ERROR",
        "service": "auto-assign-service",
        "message": "Auto-assign selected user_id=4455 (Maria Chen) for task TF-1210 but user has 34 open tasks. Workload threshold is 30. Stale workload cache suspected — last refresh 22 minutes ago.",
        "trace_id": "assign-004",
        "request_id": "req-77d1"
    },
    {
        "timestamp": "2026-05-05T16:45:13Z",
        "level": "WARN",
        "service": "auto-assign-service",
        "message": "Workload cache age: 22m (expected max: 15m). Redis queue backlog: 847 jobs. Possible queue congestion.",
        "trace_id": "assign-004",
        "request_id": "req-77d2"
    },
    {
        "timestamp": "2026-05-06T08:30:00Z",
        "level": "ERROR",
        "service": "webhook-service",
        "message": "Slack webhook delivery failed: 3/3 retries exhausted. webhook_id=wh_89, channel=#eng-updates, error=403 Forbidden. Slack app authorization may have been revoked.",
        "trace_id": "wh-005",
        "request_id": "req-88e1"
    },
    {
        "timestamp": "2026-05-06T08:30:01Z",
        "level": "INFO",
        "service": "webhook-service",
        "message": "Webhook wh_89 marked as 'degraded'. Admin notification sent to user_id=1001.",
        "trace_id": "wh-005",
        "request_id": "req-88e2"
    },
    {
        "timestamp": "2026-05-06T13:10:22Z",
        "level": "ERROR",
        "service": "file-service",
        "message": "File download returned 403: signed URL expired. attachment_id=att_3321, task_id=TF-1198, url_age=74m (max: 60m).",
        "trace_id": "file-006",
        "request_id": "req-99f1"
    },
    {
        "timestamp": "2026-05-07T09:01:15Z",
        "level": "ERROR",
        "service": "digest-service",
        "message": "Weekly digest generation timeout for project proj_18: exceeded 300s limit. Project has 412 modified tasks in past 7 days. Falling back to category summary.",
        "trace_id": "dig-007",
        "request_id": "req-aag1"
    },
    {
        "timestamp": "2026-05-07T09:01:16Z",
        "level": "INFO",
        "service": "digest-service",
        "message": "Digest for proj_18 completed in 287s after category fallback. Delivered to 14 recipients via email. Slack delivery pending.",
        "trace_id": "dig-007",
        "request_id": "req-aag2"
    },
    {
        "timestamp": "2026-05-06T15:33:40Z",
        "level": "ERROR",
        "service": "dashboard-service",
        "message": "Velocity chart returned 'No data available' for project proj_05. Cause: 0/47 tasks in current sprint have story_points field populated.",
        "trace_id": "dash-008",
        "request_id": "req-bbh1"
    }
]

print(f"Loaded {len(ERROR_LOGS)} error log entries")

In [ ]:
# --- Past Issues ---
PAST_ISSUES = [
    {
        "id": "BUG-234",
        "title": "Export fails with 413 for datasets over 10K rows",
        "status": "resolved",
        "resolution": "This is by design — the /api/v2/export endpoint has a 10K row limit. Users should use /api/v2/export/bulk for larger datasets. Updated error message to include a link to the bulk endpoint documentation.",
        "date_resolved": "2026-03-10",
        "tags": ["export", "api", "413"]
    },
    {
        "id": "BUG-267",
        "title": "AI summarizer includes deleted tasks in summary",
        "status": "open",
        "severity": "medium",
        "description": "The summarizer processes all comments in a task thread but does not check whether referenced tasks still exist. This causes summaries to include action items that reference deleted tasks.",
        "date_opened": "2026-04-15",
        "tags": ["ai", "summarizer", "deleted-tasks"]
    },
    {
        "id": "BUG-189",
        "title": "SSO redirect loop with Okta when ACS URL uses HTTP",
        "status": "resolved",
        "resolution": "Root cause was customers configuring the ACS URL with http:// instead of https://. Added a validation check during SSO setup that rejects non-HTTPS ACS URLs. Also added a specific error message for redirect loops suggesting the admin check the ACS URL protocol.",
        "date_resolved": "2026-02-28",
        "tags": ["sso", "okta", "redirect", "authentication"]
    },
    {
        "id": "BUG-301",
        "title": "Auto-assign overloads team members when Redis queue backs up",
        "status": "open",
        "severity": "high",
        "description": "When the Redis job queue has high backlog (800+ jobs), the workload cache refresh exceeds the 15-minute window. Auto-assign then uses stale data and may assign tasks to overloaded team members. Occurs during peak usage hours (9-11 AM).",
        "date_opened": "2026-05-01",
        "tags": ["auto-assign", "redis", "workload", "cache"]
    },
    {
        "id": "BUG-156",
        "title": "File download links expire after 1 hour",
        "status": "resolved",
        "resolution": "By design — S3 signed URLs expire after 60 minutes for security. Reloading the task page generates a fresh URL. Added a user-facing message: 'Link expired — click to refresh.'",
        "date_resolved": "2026-01-20",
        "tags": ["files", "s3", "download", "expired"]
    },
    {
        "id": "BUG-278",
        "title": "Dashboard velocity chart shows 'No data' when story points missing",
        "status": "resolved",
        "resolution": "Velocity chart requires story points on tasks. Added an onboarding tooltip and a fallback to show task-count-based velocity when no story points exist.",
        "date_resolved": "2026-04-02",
        "tags": ["dashboard", "velocity", "story-points"]
    },
    {
        "id": "BUG-312",
        "title": "Slack webhook stops working after Slack workspace admin revokes app",
        "status": "resolved",
        "resolution": "Expected behavior when Slack admin revokes OAuth. Improved detection: webhook now shows 'Authorization revoked — reconnect in Settings > Integrations' instead of generic 'delivery failed' message.",
        "date_resolved": "2026-04-20",
        "tags": ["webhook", "slack", "oauth", "authorization"]
    }
]

print(f"Loaded {len(PAST_ISSUES)} past issues")

In [ ]:
# --- Test Coverage Registry ---
TEST_REGISTRY = {
    "export_api": {
        "file": "export.spec.ts",
        "last_run": "passing",
        "last_run_date": "2026-05-06",
        "scenarios": [
            "CSV export under 10K rows",
            "JSON export under 10K rows",
            "413 error for over 10K rows",
            "Guest user receives 403",
            "Bulk export generates download link"
        ]
    },
    "sso_authentication": {
        "file": "sso.spec.ts",
        "last_run": "passing",
        "last_run_date": "2026-05-06",
        "scenarios": [
            "Okta SSO happy path",
            "Azure AD SSO happy path",
            "Invalid ACS URL returns error",
            "Session expiry after configured duration"
        ]
    },
    "ai_summarizer": {
        "file": "ai-summarizer.spec.ts",
        "last_run": "failing",
        "last_run_date": "2026-05-06",
        "scenarios": [
            "Summarize task with 5+ comments",
            "Guest user cannot access summarizer",
            "Summary under 200 words",
            "FAILING: Summary does not reference deleted tasks"
        ]
    },
    "auto_assign": {
        "file": "auto-assign.spec.ts",
        "last_run": "passing",
        "last_run_date": "2026-05-05",
        "scenarios": [
            "Suggest mode shows recommendation",
            "Auto mode assigns on creation",
            "No assignee when no match found",
            "Respects workload threshold"
        ]
    },
    "webhooks": {
        "file": "webhooks.spec.ts",
        "last_run": "passing",
        "last_run_date": "2026-05-06",
        "scenarios": [
            "Slack webhook fires on task creation",
            "Retry logic on delivery failure",
            "Webhook marked degraded after 3 failures",
            "GitHub webhook payload format"
        ]
    },
    "file_attachments": {
        "file": "attachments.spec.ts",
        "last_run": "passing",
        "last_run_date": "2026-05-06",
        "scenarios": [
            "Upload file under 50MB",
            "Reject file over 50MB",
            "Signed URL generation",
            "Preview for supported formats"
        ]
    }
}

print(f"Loaded {len(TEST_REGISTRY)} test registry entries")

## 5. Tool Functions

Simple keyword-matching search over the simulated data.

In [ ]:
def search_product_docs(query: str) -> str:
    """Search product documentation by keyword."""
    query_lower = query.lower()
    results = []
    for key, doc in PRODUCT_DOCS.items():
        searchable = (doc["title"] + " " + doc["content"] + " " + " ".join(doc["tags"])).lower()
        if any(term in searchable for term in query_lower.split()):
            results.append({
                "id": key,
                "title": doc["title"],
                "content": doc["content"],
                "last_updated": doc["last_updated"]
            })
    if not results:
        return f"No documentation found for query: '{query}'"
    return json.dumps(results, indent=2)


def pull_error_logs(service: str = None, keyword: str = None) -> str:
    """Pull error/warning logs, optionally filtered by service or keyword."""
    results = []
    for log in ERROR_LOGS:
        if service and service.lower() not in log["service"].lower():
            continue
        if keyword and keyword.lower() not in log["message"].lower():
            continue
        results.append(log)
    if not results:
        filter_desc = ""
        if service:
            filter_desc += f" service='{service}'"
        if keyword:
            filter_desc += f" keyword='{keyword}'"
        return f"No logs found matching:{filter_desc}"
    return json.dumps(results, indent=2)


def search_past_issues(query: str) -> str:
    """Search past issues by keyword."""
    query_lower = query.lower()
    results = []
    for issue in PAST_ISSUES:
        searchable = (issue["title"] + " " + " ".join(issue["tags"])).lower()
        if "resolution" in issue:
            searchable += " " + issue["resolution"].lower()
        if "description" in issue:
            searchable += " " + issue["description"].lower()
        if any(term in searchable for term in query_lower.split()):
            results.append(issue)
    if not results:
        return f"No past issues found for query: '{query}'"
    return json.dumps(results, indent=2)


def check_test_coverage(feature_area: str) -> str:
    """Check e2e test coverage for a feature area."""
    if feature_area in TEST_REGISTRY:
        return json.dumps(TEST_REGISTRY[feature_area], indent=2)
    for key, entry in TEST_REGISTRY.items():
        if feature_area.lower().replace(" ", "_") in key or key in feature_area.lower().replace(" ", "_"):
            return json.dumps(entry, indent=2)
    return f"No test coverage data found for feature area: '{feature_area}'"


TOOL_FUNCTIONS = {
    "search_product_docs": search_product_docs,
    "pull_error_logs": pull_error_logs,
    "search_past_issues": search_past_issues,
    "check_test_coverage": check_test_coverage,
}

print("Tool functions ready:", list(TOOL_FUNCTIONS.keys()))

## 6. Tool Schemas & System Prompt

In [ ]:
TOOLS = [
    {
        "name": "search_product_docs",
        "description": (
            "Search TaskFlow AI's product documentation and knowledge base. "
            "Use this to understand intended product behavior, feature specifications, "
            "known limitations, and configuration requirements. "
            "Returns matching documentation entries."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query — use keywords related to the feature or behavior in question"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "pull_error_logs",
        "description": (
            "Pull recent error and warning logs from TaskFlow AI's services. "
            "Use this to find error traces, stack traces, and system warnings "
            "related to a reported issue. Can filter by service name or keyword."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "service": {
                    "type": "string",
                    "description": "Filter by service name (e.g., 'auth-service', 'export-service', 'ai-summarizer', 'auto-assign-service', 'webhook-service', 'file-service', 'digest-service', 'dashboard-service')"
                },
                "keyword": {
                    "type": "string",
                    "description": "Filter logs containing this keyword"
                }
            },
            "required": []
        }
    },
    {
        "name": "search_past_issues",
        "description": (
            "Search the issue tracker for previously reported bugs and their resolutions. "
            "Use this to check if a reported issue is a known bug, has been resolved before, "
            "or is a duplicate of an existing issue."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query — use keywords describing the issue"
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "check_test_coverage",
        "description": (
            "Check whether there is existing end-to-end test coverage for a "
            "given feature area. Returns test file name, last run status, and "
            "what scenarios are covered."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "feature_area": {
                    "type": "string",
                    "description": "The feature area to check (e.g., 'export_api', 'sso_authentication', 'ai_summarizer')"
                }
            },
            "required": ["feature_area"]
        }
    }
]

SYSTEM_PROMPT = """You are a Bug Triage Agent for TaskFlow AI, a project management SaaS platform.

Your job is to investigate bug reports submitted by team members and classify them accurately.

For each bug report, you should:
1. Search product documentation to understand the expected behavior of the feature in question.
2. Pull error logs to find any system errors or warnings related to the report.
3. Search past issues to check if this is a known bug, a duplicate, or was previously resolved.
4. Optionally check test coverage to see if the affected area has automated tests.

After investigating, classify the report into one of these categories:
- **bug**: A genuine software defect that needs engineering attention.
- **intended_behavior**: The system is working as designed. The user may need guidance or the UX may need improvement, but there is no code defect.
- **user_error**: The user is doing something incorrectly or lacks the right permissions/configuration. Provide guidance on the correct approach.
- **needs_more_info**: The report is too vague to investigate. Specify what additional information is needed.
- **duplicate**: This matches an existing open or recently resolved issue.

For each classification, provide:
- A confidence level (high, medium, low)
- A brief summary of your investigation findings
- The evidence that led to your classification (which docs, logs, or past issues informed your decision)
- A recommended next step (e.g., \"Guide client to use bulk export endpoint\" or \"Escalate to engineering — open bug BUG-301\")
- If a duplicate, reference the matching issue ID

Be thorough but concise. Your investigation will be posted back to the bug report thread for the team to review."""

print("Tools and system prompt ready")

## 7. Agent Loop

Each call to `triage_bug_report` creates a parent `AGENT` span. Claude's LLM calls and each tool execution are nested as child spans underneath it, producing a single trace per report in Arize.

In [ ]:
def call_llm(messages: list, iteration: int) -> object:
    """Call Claude, wrapped in a manual LLM span that nests under the active parent."""
    with tracer.start_as_current_span(f"messages.create (iter {iteration})") as span:
        span.set_attribute("openinference.span.kind", "LLM")
        span.set_attribute("llm.model_name", MODEL)
        span.set_attribute("input.value", json.dumps(messages[-1]["content"] if messages else ""))

        response = client.messages.create(
            model=MODEL,
            max_tokens=4096,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages
        )

        output_text = ""
        for block in response.content:
            if hasattr(block, "text"):
                output_text = block.text
                break
            elif block.type == "tool_use":
                output_text = f"tool_use: {block.name}({json.dumps(block.input)})"
                break

        span.set_attribute("output.value", output_text)
        span.set_status(Status(StatusCode.OK))
        return response


def run_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch a tool call, wrapped in a TOOL span that nests under the active parent."""
    with tracer.start_as_current_span(tool_name) as span:
        span.set_attribute("openinference.span.kind", "TOOL")
        span.set_attribute("tool.name", tool_name)
        span.set_attribute("input.value", json.dumps(tool_input))

        fn = TOOL_FUNCTIONS.get(tool_name)
        result = fn(**tool_input) if fn else f"Error: unknown tool '{tool_name}'"

        span.set_attribute("output.value", result[:1000] if len(result) > 1000 else result)
        span.set_status(Status(StatusCode.OK))
        return result


def triage_bug_report(report: dict, verbose: bool = True) -> dict:
    """
    Run the bug triage agent on a single report.
    Parent AGENT span wraps all LLM and TOOL child spans.
    """
    report_id = report["id"]
    reporter  = report["reporter"]
    message   = report["message"]

    with tracer.start_as_current_span(f"triage:{report_id}") as agent_span:
        agent_span.set_attribute("openinference.span.kind", "AGENT")
        agent_span.set_attribute("input.value", message)
        agent_span.set_attribute("report.id", report_id)
        agent_span.set_attribute("report.reporter", reporter)

        if verbose:
            print(f"\n{'='*60}")
            print(f"Report: {report_id} | Reporter: {reporter}")
            print(f"Message: {message[:120]}..." if len(message) > 120 else f"Message: {message}")
            print(f"{'='*60}")

        messages = [{"role": "user", "content": f"Bug report from {reporter}:\n\n{message}"}]

        tools_called = []
        final_response = None
        iteration = 0

        while iteration < 10:
            iteration += 1
            response = call_llm(messages, iteration)

            if verbose:
                print(f"\n[Iteration {iteration}] stop_reason={response.stop_reason}")

            if response.stop_reason == "end_turn":
                for block in response.content:
                    if hasattr(block, "text"):
                        final_response = block.text
                        break
                break

            if response.stop_reason == "tool_use":
                messages.append({"role": "assistant", "content": response.content})
                tool_results = []
                for block in response.content:
                    if block.type == "tool_use":
                        if verbose:
                            print(f"  -> Tool: {block.name}({json.dumps(block.input)})")
                        result = run_tool(block.name, block.input)
                        tools_called.append({"tool": block.name, "input": block.input})
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": result
                        })
                messages.append({"role": "user", "content": tool_results})
            else:
                break

        if final_response:
            agent_span.set_attribute("output.value", final_response)
        agent_span.set_attribute("agent.iterations", iteration)
        agent_span.set_attribute("agent.tools_called", json.dumps([t["tool"] for t in tools_called]))
        agent_span.set_status(Status(StatusCode.OK))

        if verbose:
            print(f"\nTools called ({len(tools_called)}): {[t['tool'] for t in tools_called]}")
            print(f"\n--- Final Investigation ---")
            print(final_response)

        return {
            "report_id": report_id,
            "reporter": reporter,
            "tools_called": tools_called,
            "final_response": final_response,
            "iterations": iteration
        }


print("Agent loop ready")

## 8. Test Bug Reports

In [ ]:
TEST_BUG_REPORTS = [
    {
        "id": "report-1",
        "reporter": "Sarah (Account Manager)",
        "message": (
            "Client Acme Corp says they can't export their task data. They're getting "
            "an error when trying to download a CSV from the API. They have about 25,000 "
            "tasks and need this for their quarterly review ASAP."
        ),
        "ground_truth": {
            "classification": "intended_behavior",
            "reasoning": "The export API has a 10K row limit. Client needs to use the bulk export endpoint. This is a known, documented limitation.",
            "expected_tools": ["search_product_docs", "pull_error_logs", "search_past_issues"],
            "similar_issue": "BUG-234"
        }
    },
    {
        "id": "report-2",
        "reporter": "Jake (Engineer)",
        "message": (
            "The AI task summarizer is generating summaries that mention task TF-892 "
            "with action items, but TF-892 was deleted last week. A customer noticed "
            "this in their summary for TF-1204 and is confused about why they're being "
            "told to follow up on a task that doesn't exist."
        ),
        "ground_truth": {
            "classification": "bug",
            "reasoning": "The summarizer does not check whether referenced tasks still exist. This is a known open bug (BUG-267).",
            "expected_tools": ["search_product_docs", "pull_error_logs", "search_past_issues"],
            "similar_issue": "BUG-267"
        }
    },
    {
        "id": "report-3",
        "reporter": "Tom (Customer Success)",
        "message": (
            "New customer Pinnacle Inc just set up SSO with Okta but their users "
            "are getting stuck in a login loop. They click login, get redirected to "
            "Okta, authenticate, and then get sent back to the login page again. "
            "This is blocking their entire team from accessing TaskFlow."
        ),
        "ground_truth": {
            "classification": "user_error",
            "reasoning": "Most likely an HTTP/HTTPS mismatch in the ACS URL. Error logs confirm http:// instead of https://.",
            "expected_tools": ["search_product_docs", "pull_error_logs", "search_past_issues"],
            "similar_issue": "BUG-189"
        }
    },
    {
        "id": "report-4",
        "reporter": "Lisa (Engineering Manager)",
        "message": (
            "Smart auto-assign keeps putting tasks on Maria even though she's already "
            "overloaded. She has like 30+ open tasks and the system just assigned her "
            "another one. This is happening during our morning standup time when everyone "
            "is creating tasks."
        ),
        "ground_truth": {
            "classification": "bug",
            "reasoning": "The workload cache is stale due to Redis queue congestion. Known open bug BUG-301.",
            "expected_tools": ["search_product_docs", "pull_error_logs", "search_past_issues"],
            "similar_issue": "BUG-301"
        }
    },
    {
        "id": "report-5",
        "reporter": "Amy (Designer)",
        "message": (
            "I can't download an image I attached to a task yesterday. When I click "
            "the download link it gives me an access denied error. I definitely have "
            "permission to see this task — I'm the one who created it."
        ),
        "ground_truth": {
            "classification": "intended_behavior",
            "reasoning": "S3 signed URLs expire after 1 hour. Reloading the task generates a fresh URL.",
            "expected_tools": ["search_product_docs", "pull_error_logs", "search_past_issues"],
            "similar_issue": "BUG-156"
        }
    },
    {
        "id": "report-6",
        "reporter": "Kevin (Intern)",
        "message": (
            "I'm trying to use the AI summarizer on a task but nothing happens when "
            "I click the Summarize button. The other features seem to work fine. "
            "I just joined the team last week."
        ),
        "ground_truth": {
            "classification": "user_error",
            "reasoning": "Kevin is likely a Guest user. Guests cannot use AI features.",
            "expected_tools": ["search_product_docs"],
            "similar_issue": None
        }
    },
    {
        "id": "report-7",
        "reporter": "Priya (Tech Lead)",
        "message": (
            "Our Slack notifications from TaskFlow stopped working sometime this week. "
            "We're not getting any updates in #eng-updates anymore. I checked and the "
            "integration still shows as 'connected' in our settings but the status "
            "says 'degraded'. What does that mean?"
        ),
        "ground_truth": {
            "classification": "user_error",
            "reasoning": "Webhook degraded because Slack admin revoked OAuth app. Fix: re-authorize in Settings > Integrations.",
            "expected_tools": ["search_product_docs", "pull_error_logs", "search_past_issues"],
            "similar_issue": "BUG-312"
        }
    },
    {
        "id": "report-8",
        "reporter": "David (VP Engineering)",
        "message": (
            "The weekly AI digest for Project Phoenix was really late this Monday — "
            "didn't arrive until almost 9:30 AM instead of 9. Also it was way less "
            "detailed than usual, just high-level categories instead of listing "
            "individual tasks. We rely on this for our Monday standup."
        ),
        "ground_truth": {
            "classification": "intended_behavior",
            "reasoning": "412 modified tasks triggered the 300s timeout and category fallback. Documented behavior.",
            "expected_tools": ["search_product_docs", "pull_error_logs"],
            "similar_issue": None
        }
    }
]

print(f"{len(TEST_BUG_REPORTS)} test reports loaded")

## 9. Run Report 1 — Export 413 Error

**Expected:** `intended_behavior` — 10K row limit, client should use bulk export.  
**Expected tools:** `search_product_docs` → `pull_error_logs` → `search_past_issues`

In [ ]:
result_1 = triage_bug_report(TEST_BUG_REPORTS[0], verbose=True)

In [ ]:
# Compare to ground truth
gt = TEST_BUG_REPORTS[0]["ground_truth"]
print("Ground Truth Classification:", gt["classification"])
print("Ground Truth Reasoning:", gt["reasoning"])
print("Expected Tools:", gt["expected_tools"])
print("\nActual Tools Called:", [t["tool"] for t in result_1["tools_called"]])

## 10. Run All 8 Reports

In [ ]:
all_results = []

for report in TEST_BUG_REPORTS:
    result = triage_bug_report(report, verbose=False)
    all_results.append(result)

print(f"Completed {len(all_results)} reports")

In [ ]:
print(f"{'Report':<12} {'Tools Called':<55} {'Iterations'}")
print("-" * 75)
for r in all_results:
    tools_str = ", ".join([t["tool"].replace("search_","s_").replace("pull_","p_").replace("check_","c_") for t in r["tools_called"]])
    print(f"{r['report_id']:<12} {tools_str:<55} {r['iterations']}")

Report       Tools Called                                            Iterations
---------------------------------------------------------------------------
report-1     s_product_docs, p_error_logs, s_past_issues, c_test_coverage 3
report-2     s_product_docs, p_error_logs, s_past_issues, c_test_coverage 2
report-3     s_product_docs, p_error_logs, s_past_issues, c_test_coverage 2
report-4     s_product_docs, p_error_logs, s_past_issues, c_test_coverage 3
report-5     s_product_docs, p_error_logs, s_past_issues, c_test_coverage, p_error_logs 3
report-6     s_product_docs, p_error_logs, s_past_issues, c_test_coverage 2
report-7     s_product_docs, p_error_logs, s_past_issues, c_test_coverage 3
report-8     s_product_docs, s_product_docs, p_error_logs, p_error_logs, s_past_issues, s_past_issues, c_test_coverage 3


## 11. Eval Templates

Templates for running LLM-as-judge evaluations in Arize.

In [ ]:
CLASSIFICATION_EVAL_TEMPLATE = """You are evaluating a Bug Triage Agent's classification of a bug report.

[BEGIN DATA]
***
[Bug Report]: {bug_report}
***
[Agent Classification]: {agent_classification}
***
[Ground Truth Classification]: {ground_truth_classification}
***
[Ground Truth Reasoning]: {ground_truth_reasoning}
***
[END DATA]

Determine whether the agent's classification matches the ground truth.
Consider partial credit: if the ground truth is "intended_behavior" and the agent said "user_error", this may be partially correct if the reasoning is sound.

Your response must be a single word: "correct", "partially_correct", or "incorrect"."""


TOOL_SELECTION_EVAL_TEMPLATE = """You are evaluating whether a Bug Triage Agent selected the appropriate tools to investigate a bug report.

[BEGIN DATA]
***
[Bug Report]: {bug_report}
***
[Tools Called]: {tools_called}
***
[Expected Tools]: {expected_tools}
***
[END DATA]

Evaluate whether the agent called the right tools. Consider:
- Did it call all necessary tools?
- Did it call any unnecessary tools (not harmful, but inefficient)?
- Did it skip a tool that would have provided important evidence?

Your response must be a single word: "correct", "partially_correct", or "incorrect"."""


INVESTIGATION_QUALITY_EVAL_TEMPLATE = """You are evaluating the quality of a Bug Triage Agent's investigation summary.

[BEGIN DATA]
***
[Bug Report]: {bug_report}
***
[Agent Summary]: {agent_summary}
***
[Ground Truth Reasoning]: {ground_truth_reasoning}
***
[END DATA]

Evaluate the investigation summary on these criteria:
1. Did it identify the correct root cause or explanation?
2. Is the evidence cited relevant and accurate?
3. Is the recommended next step actionable and appropriate?
4. Is the summary concise enough to be useful in a Slack thread?

Your response must be a single word: "high_quality", "acceptable", or "poor"."""


print("Eval templates ready")

## 12. Build Dataset

Combine inputs, agent outputs, and ground truth labels for upload to Arize.

In [ ]:
dataset_rows = []

for report, result in zip(TEST_BUG_REPORTS, all_results):
    gt = report["ground_truth"]
    row = {
        "report_id": report["id"],
        "reporter": report["reporter"],
        "bug_report": report["message"],
        "agent_response": result["final_response"],
        "tools_called": json.dumps([t["tool"] for t in result["tools_called"]]),
        "iterations": result["iterations"],
        "ground_truth_classification": gt["classification"],
        "ground_truth_reasoning": gt["reasoning"],
        "expected_tools": json.dumps(gt["expected_tools"]),
        "similar_issue": gt.get("similar_issue", ""),
    }
    dataset_rows.append(row)

print(f"Dataset prepared: {len(dataset_rows)} rows")

with open("triage_dataset.json", "w") as f:
    json.dump(dataset_rows, f, indent=2)

print("Saved to triage_dataset.json")